In [ ]:
import numpy as np
import pandas as pd

#Seteando tamaño max de columnas y registros
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.options.display.float_format = '{:.2f}'.format

import matplotlib.pyplot as plt
import emoji
import os
import time
from glob import glob

import csv, re, html, emoji, fasttext
from contractions import fix
from nltk.corpus import stopwords
import spacy

from ekphrasis.classes.preprocessor import TextPreProcessor
from ekphrasis.dicts.emoticons import emoticons
from ekphrasis.dicts.noslang.slangdict import slangdict

### Concatenate all together

In [ ]:
def ObtenerArchivos(ruta_actual,carpeta):
    """Esta función nos devuelve una lista con los archivos de una carpeta"""
    ruta_completa = os.path.join(ruta_actual,carpeta)
    archivos = glob(ruta_completa+"/*")
    return archivos

In [ ]:
files = ObtenerArchivos(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/",
                        carpeta="list_scrap")
files

In [ ]:
lista = [] 
for archivo in files:
    if ".csv" in archivo:
        print("archivo: ",archivo)
        df = pd.read_csv(archivo,
                         sep="|")
        df = df[['id','author','body','body_html']]
        df['topic'] = archivo.split('/')[-1].replace('_all.csv', '')
        lista.append(df)
        
df_all = pd.concat(lista,axis=0, sort = False)
df_all

In [ ]:
df_all.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/list_scrap/all.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
              )

### All in 1 DF

In [ ]:
df_all = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/all.csv",
                     sep="|"
                     )
df_all

In [ ]:
df_all[df_all['id']=="hvzim0m"]

### ID repetidos

In [ ]:
prueba = pd.DataFrame(df_all['id'].value_counts()).reset_index()
prueba = prueba[prueba['count']>1].reset_index()
prueba

In [ ]:
prueba.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/ids_repetidos.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
             )

In [ ]:
prueba = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/ids_repetidos.csv",
                     sep="|"
                     )
prueba

In [ ]:
prueba['id'].sum()

In [ ]:
488871-243347 # Cantidad eliminada

#### W/O duplicates

In [ ]:
df_all

In [ ]:
df_all[df_all['id']=='i0eirx4']

In [ ]:
df_all[df_all.duplicated(subset=['id','author','body','body_html'])]

In [ ]:
len(df_all)-245524

In [ ]:
df_all.drop_duplicates(subset=['id','author','body','body_html'],keep='first', inplace=True)

In [ ]:
len(df_all)

In [ ]:
len(df_all[df_all.duplicated(subset=['id','author','body'])])

#### Removing NULL comments

In [ ]:
len(df_all[df_all['body'].isnull()])

In [ ]:
12095409-78

In [ ]:
df_all.dropna(subset=['body'], inplace=True)
len(df_all)

In [ ]:
# NOT DUPLICATED OR NULL COMMENTS
df_all.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/all_nd_nn.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
              )

### Cleansing

In [ ]:
df_all = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/all_nd_nn.csv",
                     sep="|"
                     )
df_all

In [ ]:
df_all.drop(columns=['body_html'], inplace=True)
df_all

In [ ]:
12340933-12095409 # OK:coincide con cantidad eliminada

In [ ]:
# Create the preprocessor (NOT USED, just internal dicts)
text_processor = TextPreProcessor(
    unpack_hashtags=True
)

In [ ]:
def expand_slang(text, slang_dict):
    # Match words only (case-sensitive)
    def replace(match):
        word = match.group(0)
        return slang_dict.get(word, word)  
    return re.sub(r'\b\w+\b', replace, text)

In [ ]:
slangdict_with_upper = {**slangdict, **{k.upper(): v for k, v in slangdict.items()}}
slangdict_with_upper = {**slangdict_with_upper, **{k.capitalize(): v for k, v in slangdict.items()}}

In [ ]:
def clean_text(text):
    # Remove emojis using regex
    def remove_emojis_regex(text):
        emoji_pattern = re.compile(
                                    '['
                                    '\U0001F600-\U0001F64F'
                                    '\U0001F300-\U0001F5FF'
                                    '\U0001F680-\U0001F6FF'
                                    '\U0001F700-\U0001F77F'
                                    '\U0001F780-\U0001F7FF'
                                    '\U0001F800-\U0001F8FF'
                                    '\U0001F900-\U0001F9FF'
                                    '\U0001FA00-\U0001FA6F'
                                    '\U0001FA70-\U0001FAFF'
                                    '\U00002700-\U000027BF'
                                    '\U000024C2-\U0001F251'
                                    ']+', flags=re.UNICODE
                                   )
        return emoji_pattern.sub(' ', text)

    
    text = "".join(text_processor.pre_process_doc(text))
    #text = emoji.replace_emoji(text, replace='')          # Remove emojis
    text = remove_emojis_regex(text)                       # Extra emoji removal
    text = html.unescape(text)
    
    try:
        text = fix(text)                                   # Expand contractions
    except Exception as e:
        print(text)
    
    text = re.sub(r'http\S+|www.\S+|https\S+', '', text)   # Remove URLs
    text = re.sub(r'@\w+', '', text)                       # Remove mentions
    #text = re.sub(r'#(\w+)', r'\1', text)                 # Remove '#' from hashtags
    text = expand_slang(text, slangdict_with_upper)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)             # Reduce character repetition
    text = re.sub(r"[^A-Za-z0-9\s.,'!?-]", '', text)       # Remove unwanted symbols
    text = re.sub(r'\s+', ' ', text).strip()               # Normalize whitespace

    # Replace control and formatting characters
    text = re.sub(r'[\n\t\r\f\v\b\a\0]', '', text)
    text = re.sub(r'[①②③④⑤⑥⑦⑧⑨⑩]', '', text)
    text = re.sub(r'[\x00-\x1F]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:
text = "OMG THIS is soooo    coooool! He's a Shangri-la ②⑤③ %$ @varsayoy 😂 lol Carlo's #BestDayEver https://t.co/xyz"
clean_text(text)

In [ ]:
# USING REGEX
def remove_emojis_regex(text):
    # Regular expression pattern to match emoji
    emoji_pattern = re.compile(
                                "[" 
                                "\U0001F600-\U0001F64F"  # Emoticons
                                "\U0001F300-\U0001F5FF"  # Symbols & Pictographs
                                "\U0001F680-\U0001F6FF"  # Transport & Map symbols
                                "\U0001F700-\U0001F77F"  # Alchemical symbols
                                "\U0001F780-\U0001F7FF"  # Geometric shapes
                                "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
                                "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
                                "\U0001FA00-\U0001FA6F"  # Chess Symbols
                                "\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
                                "\U00002700-\U000027BF"  # Dingbats
                                "\U000024C2-\U0001F251" 
                                "]+", flags=re.UNICODE
                                )
    return emoji_pattern.sub(r' ', text)

# Apply the function to the text column
df_all['body_ne'] = df_all['body'].astype(str).apply(remove_emojis_regex)
df_all

In [ ]:
# USING emoji LIBRARY
def remove_emojis(text):
    return emoji.replace_emoji(text, replace=' ')

# Apply the function to the text column
df_all['body_ne'] = df_all['body_ne'].apply(remove_emojis)
df_all

### Removing Special Characters

In [ ]:
# USING REGEX
# Remove all occurrences of these characters
pattern = r'[\n\t\r\f\v\b\a\0]'
df_all['body_ne_nsc'] = df_all['body_ne'].str.replace(pattern, ' ', regex=True)

# Remove all occurrences of these characters 2
pattern = r'[①②③④⑤⑥⑦⑧⑨⑩]'
df_all['body_ne_nsc'] = df_all['body_ne_nsc'].str.replace(pattern, ' ', regex=True)

# Remove all occurrences of these characters 3
pattern = r'[\x00-\x1F]'
df_all['body_ne_nsc'] = df_all['body_ne_nsc'].str.replace(pattern, ' ', regex=True)

df_all['body_ne_nsc'] = df_all['body_ne_nsc'].str.strip()

df_all.drop(columns=['body_ne'], inplace=True)
df_all

### Remove all non-alphanumeric

In [ ]:
# USING REGEX
# remove all non-alphanumeric characters (including superscripts, special chars)
df_all['body_ne_nsc'] = df_all['body_ne_nsc'].str.replace(r'[^a-zA-Z0-9\s.,-:!?]', ' ', regex=True)

# remove multiple spaces 
df_all['body_ne_nsc'] = df_all['body_ne_nsc'].str.replace(r'\s+', ' ', regex=True)

df_all['body_ne_nsc'] = df_all['body_ne_nsc'].str.strip()

df_all

### New version using clean_text function

In [ ]:
df_all['body_cleand'] = df_all['body'].apply(lambda x: clean_text(x))
df_all

In [ ]:
# for validating errors in especific comments
for i, row in df_all.iterrows():
    try:
        fix(row['body'])
    except Exception as e:
        print(f"Row {i} failed: {e} | Text: {row['body']}")

### Comments length

In [ ]:
df_all['body_len'] = df_all['body_cleand'].str.len()
df_all

In [ ]:
df_all['body_len'].describe()

In [ ]:
# NOT DUPLICATED OR NULL COMMENTS, NO emojis, NO SPECIAL CHAR
df_all.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/all_nd_nn_ne_nsc.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
              )

### Words Count

In [ ]:
df_all["word_count"] = df_all["body_cleand"].fillna("").str.split().apply(len)
df_all

### Comments' Language 

In [ ]:
df_all = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/all_nd_nn_ne_nsc.csv",
                     sep="|"
                     )
df_all.rename(columns={"body_ne_nsc":"body_cleand"}, inplace=True)
df_all

In [ ]:
# Load the language identification model
model = fasttext.load_model('/home/varsayou/Downloads/lid.176.bin')
#model = fasttext.load_model('/Proyecto/Value-disagreement/Modelos/lid.176.bin')

In [ ]:
# Predict the language of a given text
text = "Your sample text here"
prediction = model.predict(text)
print(prediction)

In [ ]:
df_all['lang'] = df_all['body_cleand'].apply(lambda x: model.predict(str(x).replace('\n', ''), k=1))#[0][0])
df_all

In [ ]:
df_all['language'], df_all['language_porc'] = zip(*df_all.lang)
df_all

In [ ]:
df_all.drop(columns=['lang'], inplace=True)
df_all

In [ ]:
# Determine the maximum tuple length
max_length = df_all['language'].apply(len).max()

# Create new columns, filling missing values with None
df_all[[f'col{i+1}' for i in range(max_length)]] = pd.DataFrame(df_all['language'].tolist(), index=df_all.index)

df_all.drop(columns=['language'], inplace=True)
df_all.rename(columns={"col1":"language"}, inplace=True)
df_all

In [ ]:
# Determine the maximum tuple length
max_length = df_all['language_porc'].apply(len).max()

# Create new columns, filling missing values with None
df_all[[f'coll{i+1}' for i in range(max_length)]] = pd.DataFrame(df_all['language_porc'].tolist(), index=df_all.index)

df_all.drop(columns=['language_porc'], inplace=True)
df_all.rename(columns={"coll1": "language_porc"}, inplace=True)
df_all

In [ ]:
df_all.dtypes

In [ ]:
df_all['language'].unique()

In [ ]:
df_all.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/all_lang.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
              )

In [ ]:
len(df_all[df_all['language'].isin(['__label__en','__label__uk'])])

In [ ]:
12095331-11707268

In [ ]:
df_all = df_all[df_all['language'].isin(['__label__en','__label__uk'])]
df_all

In [ ]:
df_all = df_all[~df_all['body_cleand'].isna()]
df_all

In [ ]:
df_all.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/all_eng.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
              )

In [ ]:
df_all = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/all_eng.csv",
                     sep="|"
                     )
df_all

In [ ]:
df_all['language'].unique()

In [ ]:
df_all['language'].value_counts()

In [ ]:
df_all['language_porc'].describe()

In [ ]:
df_all[df_all['language_porc']<0.80]

### Repeated Comments

In [ ]:
comments_repeated = pd.DataFrame(df_all.groupby(['author','body_cleand'])['id'].nunique())
comments_repeated.reset_index(inplace=True)
comments_repeated

In [ ]:
comments_repeated.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/comments.csv",
                         header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
                         )

In [ ]:
comments_repeated = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/comments.csv",
                                sep="|"
                                )
comments_repeated

In [ ]:
comments_repeated[:200]['body_ne_nsc'].values

In [ ]:
comments_repeated.id.describe()

In [ ]:
comments_repeated.sort_values(by='id', ascending=False, inplace=True, ignore_index=True)
#comments_repeated.head(200)
#comments_repeated.iloc[600:800]
#comments_repeated[comments_repeated['id']>1]['id'].sum()
comments_repeated = comments_repeated[comments_repeated['id']>1]
comments_repeated

In [ ]:
comments_repeated['id'].sum()

In [ ]:
df_all[df_all['body_cleand']=='Thanks!']#['topic'].unique()

In [ ]:
comments_repeated.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/comments_to_del.csv",
                         header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
                         )

In [ ]:
comments_repeated = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/comments_to_del.csv",
                                sep="|"
                                )
comments_repeated

### Not common comments (just 1)

In [ ]:
df_all

In [ ]:
df_all[df_all['body_cleand']=='Thanks']#['topic'].unique()

In [ ]:
df_all[df_all.duplicated(subset=['author','body_cleand'])]

In [ ]:
len(df_all)-155955

In [ ]:
df_all.drop_duplicates(subset=['author','body_cleand'],keep='first', inplace=True)

In [ ]:
len(df_all)

In [ ]:
len(df_all[df_all.duplicated(subset=['id','author','body'])])

In [ ]:
df_all[df_all['body_cleand']=='Thanks']#['topic'].unique()

In [ ]:
df_all.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/all_eng_one.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
              )

In [ ]:
df_all = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/all_eng_one.csv",
                     sep="|"
                     )
df_all

### Checking Comments Lenthg

In [ ]:
df_all.iloc[10000:10200]

In [ ]:
df_all[df_all['body_len']==0]

In [ ]:
df_all = df_all[df_all['body_len']>0].reset_index()
len(df_all)

In [ ]:
df_all[df_all['body_cleand'].isna()]

In [ ]:
df_all = df_all[~df_all['body_cleand'].isna()]
len(df_all)

### 1 Value's word per comment

In [ ]:
dict_all = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/Dictionary/osfstorage-archive/Dictionaries/value_words_dict.csv",
                     sep="|"
                     )
dict_all

In [ ]:
# Previous version
df_all = pd.merge(df_all, dict_all, how='left', left_on='body_cleand', right_on='word')

In [ ]:
df_all

In [ ]:
df_all = df_all[(~df_all['word'].isna())|(df_all['word_count']>1)]

In [ ]:
len(df_all)

In [ ]:
df_all.drop(columns=['level_0','index','word','value_id','dic_type','value_short','value_long'], inplace=True)
df_all

In [ ]:
11551313-11364279

In [ ]:
#len(
list(df_all[df_all['word_count']==2]['body_cleand'].unique())
#)

In [ ]:
#len(df_all[df_all['body_len']<20])
df_all[df_all['body_len']<10]
#df_all = df_all[df_all['body_len']>10]
#df_all

In [ ]:
# comments by author
len(df_all['author'].unique())

In [ ]:
authors = pd.DataFrame(df_all['author'].value_counts()).reset_index()
#authors.rename(columns={"author": "count", "index": "author"}, inplace=True)
authors

In [ ]:
authors.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/authors.csv",
               header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
               )

In [ ]:
authors = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/authors.csv",
                      sep="|"
                      )
authors

In [ ]:
authors[authors['count']>200]

In [ ]:
authors['count'].describe()

In [ ]:
authors['count'].sum()

In [ ]:
df_all

In [ ]:
import pickle

with open("/Proyecto/Value-disagreement/Python/Datasets/unique_authors.pkl", "rb") as f:
    unique_authors = pickle.load(f)

In [ ]:
df_all = df_all[df_all['author'].isin(unique_authors)]

In [ ]:
df_all.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/deba_comments_final.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
              )

### Final version analysis

In [ ]:
df_all = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/deba_comments_final_all.csv",
                      sep="|"
                      )
df_all

In [ ]:
df_all[df_all['author']=='AMerrickanGirl']['id'].duplicated().any()

In [ ]:
df_all['id'].duplicated().any()

In [ ]:
len(df_all[df_all['author']=='notviccyvictor'])

In [ ]:
df_all.sample(1000).to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/deba_comments_final_sample.csv",
                           header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
                           )